# Fashion E-commerce Analytics — Data Quality

## Objective

The objective of this notebook is to identify data quality issues in the raw datasets before applying any transformation or cleaning.

The analysis focuses on:
- duplicated records
- invalid numerical values
- inconsistent dates
- missing values
- financial consistency
- referential integrity

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = Path("../data/raw")

In [3]:
customers = pd.read_csv(
    DATA_PATH / "customers.csv",
    dtype={"Telephone": "string"}
)

discounts = pd.read_csv(DATA_PATH / "discounts.csv")
employees = pd.read_csv(DATA_PATH / "employees.csv")
products = pd.read_csv(DATA_PATH / "products.csv")
stores = pd.read_csv(DATA_PATH / "stores.csv")
transactions = pd.read_csv(DATA_PATH / "transactions.csv")

In [4]:
duplicate_mask = transactions.duplicated(keep=False)

transaction_duplicates = transactions[duplicate_mask].copy()

print("Number of duplicated rows:", transactions.duplicated().sum())
print("Rows involved in duplication:", len(transaction_duplicates))

Number of duplicated rows: 798
Rows involved in duplication: 1594


In [5]:
transaction_duplicates.head(20)

,Invoice ID,Line,Customer ID,Product ID,Size,Color,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,SKU,Transaction Type,Payment Method,Invoice Total
677,RET-US-001-03558990,1,45178,2602,L,MUSTARD,25.5,1,2023-01-01 00:00:00,0.0,-15.30,1,13,USD,$,FESW2602-L-MUSTARD,Return,Credit Card,-15.30
1306,RET-US-001-03558990,1,45178,2602,L,MUSTARD,25.5,1,2023-01-01 00:00:00,0.0,-15.30,1,13,USD,$,FESW2602-L-MUSTARD,Return,Credit Card,-15.30
5965,RET-US-001-03562782,1,12413,1561,M,WHITE,35.0,1,2023-01-07 00:00:00,0.0,-21.00,1,7,USD,$,FECO1561-M-WHITE,Return,Credit Card,-21.00
5967,RET-US-001-03562782,1,12413,1561,M,WHITE,35.0,1,2023-01-07 00:00:00,0.0,-21.00,1,7,USD,$,FECO1561-M-WHITE,Return,Credit Card,-21.00
7218,RET-US-001-03562412,1,26019,2104,M,NaN,34.0,1,2023-01-08 00:00:00,0.0,-20.40,1,13,USD,$,CHSW2104-M-,Return,Cash,-20.40
7471,RET-US-001-03562412,1,26019,2104,M,NaN,34.0,1,2023-01-08 00:00:00,0.0,-20.40,1,13,USD,$,CHSW2104-M-,Return,Cash,-20.40
10629,RET-US-001-03565682,1,36512,1523,M,NaN,22.5,1,2023-01-15 00:00:00,0.0,-22.50,1,13,USD,$,MASH1523-M-,Return,Credit Card,-101.50
10642,RET-US-001-03565682,1,36512,1523,M,NaN,22.5,1,2023-01-15 00:00:00,0.0,-22.50,1,13,USD,$,MASH1523-M-,Return,Credit Card,-101.50
11392,RET-US-001-03566036,1,84951,985,G,BEIGE,49.5,1,2023-01-16 00:00:00,0.0,-49.50,1,10,USD,$,CHCO985-G-BEIGE,Return,Cash,-49.50
11407,RET-US-001-03566036,1,84951,985,G,BEIGE,49.5,1,2023-01-16 00:00:00,0.0,-49.50,1,10,USD,$,CHCO985-G-BEIGE,Return,Cash,-49.50


In [6]:
transaction_duplicates.sort_values(
    ["Invoice ID", "Line"]
).head(30)

,Invoice ID,Line,Customer ID,Product ID,Size,Color,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,SKU,Transaction Type,Payment Method,Invoice Total
1697858,RET-CN-006-02871859,1,515701,1033,XXL,NaN,117.0,1,2023-02-10 00:00:00,0.0,-117.00,6,71,CNY,¥,MAUN1033-XXL-,Return,Credit Card,-117.00
1697966,RET-CN-006-02871859,1,515701,1033,XXL,NaN,117.0,1,2023-02-10 00:00:00,0.0,-117.00,6,71,CNY,¥,MAUN1033-XXL-,Return,Credit Card,-117.00
1708166,RET-CN-006-02878641,1,361248,4656,M,NEUTRAL,409.5,1,2023-03-14 00:00:00,0.0,-409.50,6,68,CNY,¥,FESW4656-M-NEUTRAL,Return,Credit Card,-409.50
1708291,RET-CN-006-02878641,1,361248,4656,M,NEUTRAL,409.5,1,2023-03-14 00:00:00,0.0,-409.50,6,68,CNY,¥,FESW4656-M-NEUTRAL,Return,Credit Card,-409.50
1718072,RET-CN-006-02885345,1,408627,5118,M,RED,401.5,1,2023-03-30 00:00:00,0.0,-260.98,6,69,CNY,¥,CHGI5118-M-RED,Return,Cash,-260.98
1718143,RET-CN-006-02885345,1,408627,5118,M,RED,401.5,1,2023-03-30 00:00:00,0.0,-260.98,6,69,CNY,¥,CHGI5118-M-RED,Return,Cash,-260.98
1718495,RET-CN-006-02885464,1,388779,4724,M,NaN,501.5,1,2023-03-31 00:00:00,0.0,-501.50,6,70,CNY,¥,MASP4724-M-,Return,Credit Card,-501.50
1718668,RET-CN-006-02885464,1,388779,4724,M,NaN,501.5,1,2023-03-31 00:00:00,0.0,-501.50,6,70,CNY,¥,MASP4724-M-,Return,Credit Card,-501.50
1718443,RET-CN-006-02885628,1,417288,5125,S,BEIGE,409.5,1,2023-03-31 00:00:00,0.0,-266.18,6,67,CNY,¥,FEDR5125-S-BEIGE,Return,Credit Card,-266.18
1718912,RET-CN-006-02885628,1,417288,5125,S,BEIGE,409.5,1,2023-03-31 00:00:00,0.0,-266.18,6,67,CNY,¥,FEDR5125-S-BEIGE,Return,Credit Card,-266.18


In [7]:
duplicate_groups = (
    transactions
    .groupby(list(transactions.columns), dropna=False)
    .size()
    .reset_index(name="count")
)

duplicate_groups = duplicate_groups[
    duplicate_groups["count"] > 1
]

print("Number of exact duplicate groups:", len(duplicate_groups))

Number of exact duplicate groups: 796


In [8]:
duplicate_groups["count"].value_counts().sort_index()

count
2    794
3      2
Name: count, dtype: int64

In [9]:
print("Quantity <= 0:",
      (transactions["Quantity"] <= 0).sum())

Quantity <= 0: 0


In [10]:
print("Unit Price <= 0:",
      (transactions["Unit Price"] <= 0).sum())

Unit Price <= 0: 0


In [11]:
print("Line Total < 0:",
      (transactions["Line Total"] < 0).sum())

Line Total < 0: 339627


In [12]:
print("Invoice Total < 0:",
      (transactions["Invoice Total"] < 0).sum())

Invoice Total < 0: 339627


In [13]:
print("Discount < 0:",
      (transactions["Discount"] < 0).sum())

print("Discount > 1:",
      (transactions["Discount"] > 1).sum())

Discount < 0: 0
Discount > 1: 0


In [14]:
print("Production Cost <= 0:",
      (products["Production Cost"] <= 0).sum())

Production Cost <= 0: 0


In [15]:
products["Production Cost"].describe()

count    17940.000000
mean        16.096189
std         11.628072
min          0.560000
25%          7.800000
50%         13.135000
75%         20.970000
max         77.190000
Name: Production Cost, dtype: float64

In [16]:
transactions["Expected Line Total"] = (
    transactions["Unit Price"]
    * transactions["Quantity"]
    * (1 - transactions["Discount"])
)

In [17]:
transactions["Line Total Difference"] = (
    transactions["Line Total"]
    - transactions["Expected Line Total"]
).abs()

In [18]:
transactions["Line Total Difference"].describe()

count    6.416827e+06
mean     1.445479e+01
std      1.149429e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.696000e+03
Name: Line Total Difference, dtype: float64

In [19]:
financial_inconsistencies = transactions[
    transactions["Line Total Difference"] > 0.01
]

print(
    "Transactions with financial inconsistency:",
    len(financial_inconsistencies)
)

Transactions with financial inconsistency: 339627


In [20]:
financial_inconsistencies[
    [
        "Invoice ID",
        "Line",
        "Unit Price",
        "Quantity",
        "Discount",
        "Line Total",
        "Expected Line Total",
        "Line Total Difference"
    ]
].head(20)

,Invoice ID,Line,Unit Price,Quantity,Discount,Line Total,Expected Line Total,Line Total Difference
15,RET-US-001-03558764,1,75.0,1,0.0,-45.0,75.0,120.0
16,RET-US-001-03558764,2,45.5,1,0.0,-27.3,45.5,72.8
31,RET-US-001-03558763,1,95.5,1,0.0,-95.5,95.5,191.0
40,RET-US-001-03558767,1,54.0,1,0.0,-32.4,54.0,86.4
103,RET-US-001-03558799,1,42.0,1,0.0,-25.2,42.0,67.2
109,RET-US-001-03558783,1,37.0,1,0.0,-22.2,37.0,59.2
110,RET-US-001-03558783,2,61.0,1,0.0,-36.6,61.0,97.6
114,RET-US-001-03558839,1,59.5,1,0.0,-59.5,59.5,119.0
139,RET-US-001-03558841,1,24.0,1,0.0,-24.0,24.0,48.0
155,RET-US-001-03558769,1,34.0,1,0.0,-20.4,34.0,54.4


In [21]:
transactions["Date"] = pd.to_datetime(
    transactions["Date"],
    errors="coerce"
)

print("Invalid transaction dates:",
      transactions["Date"].isna().sum())

Invalid transaction dates: 0


In [22]:
print("Min transaction date:",
      transactions["Date"].min())

print("Max transaction date:",
      transactions["Date"].max())

Min transaction date: 2023-01-01 00:00:00
Max transaction date: 2025-03-18 20:59:00


In [23]:
customers["Date Of Birth"] = pd.to_datetime(
    customers["Date Of Birth"],
    errors="coerce"
)

print("Invalid birth dates:",
      customers["Date Of Birth"].isna().sum())

print("Min birth date:",
      customers["Date Of Birth"].min())

print("Max birth date:",
      customers["Date Of Birth"].max())

Invalid birth dates: 0
Min birth date: 1949-03-20 00:00:00
Max birth date: 2007-03-18 00:00:00


In [24]:
discounts["Start"] = pd.to_datetime(
    discounts["Start"],
    errors="coerce"
)

discounts["End"] = pd.to_datetime(
    discounts["End"],
    errors="coerce"
)

print("Invalid Start dates:", discounts["Start"].isna().sum())
print("Invalid End dates:", discounts["End"].isna().sum())

Invalid Start dates: 0
Invalid End dates: 0


In [25]:
invalid_discount_periods = discounts[
    discounts["End"] < discounts["Start"]
]

print(
    "Discounts with invalid periods:",
    len(invalid_discount_periods)
)

Discounts with invalid periods: 0


In [26]:
invalid_discount_periods

,Start,End,Discont,Description,Category,Sub Category


In [27]:
print("Gender:")
print(customers["Gender"].value_counts(dropna=False))

Gender:
Gender
M    964562
F    677041
D      1703
Name: count, dtype: int64


In [28]:
print("\nTransaction Type:")
print(transactions["Transaction Type"].value_counts(dropna=False))


Transaction Type:
Transaction Type
Sale      6077200
Return     339627
Name: count, dtype: int64


In [29]:
print("\nPayment Method:")
print(transactions["Payment Method"].value_counts(dropna=False))


Payment Method:
Payment Method
Credit Card    5135298
Cash           1281529
Name: count, dtype: int64


In [30]:
print("\nCurrency:")
print(transactions["Currency"].value_counts(dropna=False))


Currency:
Currency
EUR    2549397
USD    1687764
CNY    1553438
GBP     626228
Name: count, dtype: int64


In [31]:
print("\nProduct Category:")
print(products["Category"].value_counts(dropna=False))


Product Category:
Category
Feminine     7590
Masculine    6210
Children     4140
Name: count, dtype: int64


In [32]:
#Vérifier la cohérence Product ↔ Transaction
product_reference = products[
    ["Product ID", "Color"]
].rename(
    columns={"Color": "Product Color"}
)

transaction_color_check = transactions[
    ["Product ID", "Color"]
].merge(
    product_reference,
    on="Product ID",
    how="left"
)

In [33]:
transaction_color_check.head()

,Product ID,Color,Product Color
0,485,NaN,NaN
1,2779,NaN,NaN
2,64,NEUTRAL,NEUTRAL
3,131,BLUE,BLUE
4,716,WHITE,WHITE


In [36]:
missing_customers = transactions[
    ~transactions["Customer ID"].isin(customers["Customer ID"])
]

missing_products = transactions[
    ~transactions["Product ID"].isin(products["Product ID"])
]

missing_stores = transactions[
    ~transactions["Store ID"].isin(stores["Store ID"])
]

missing_employees = transactions[
    ~transactions["Employee ID"].isin(employees["Employee ID"])
]

quality_summary = {
    "Transaction rows": len(transactions),
    "Exact duplicate rows": transactions.duplicated().sum(),
    "Missing Customer IDs": len(missing_customers),
    "Missing Product IDs": len(missing_products),
    "Missing Store IDs": len(missing_stores),
    "Missing Employee IDs": len(missing_employees),
    "Invalid transaction dates": transactions["Date"].isna().sum(),
    "Invalid discount periods": len(invalid_discount_periods),
    "Financial inconsistencies": len(financial_inconsistencies)
}

quality_summary

{'Transaction rows': 6416827,
 'Exact duplicate rows': np.int64(798),
 'Missing Customer IDs': 0,
 'Missing Product IDs': 0,
 'Missing Store IDs': 0,
 'Missing Employee IDs': 0,
 'Invalid transaction dates': np.int64(0),
 'Invalid discount periods': 0,
 'Financial inconsistencies': 339627}

In [37]:
financial_inconsistencies[
    [
        "Invoice ID",
        "Line",
        "Unit Price",
        "Quantity",
        "Discount",
        "Line Total",
        "Expected Line Total",
        "Line Total Difference"
    ]
].head(20)

,Invoice ID,Line,Unit Price,Quantity,Discount,Line Total,Expected Line Total,Line Total Difference
15,RET-US-001-03558764,1,75.0,1,0.0,-45.0,75.0,120.0
16,RET-US-001-03558764,2,45.5,1,0.0,-27.3,45.5,72.8
31,RET-US-001-03558763,1,95.5,1,0.0,-95.5,95.5,191.0
40,RET-US-001-03558767,1,54.0,1,0.0,-32.4,54.0,86.4
103,RET-US-001-03558799,1,42.0,1,0.0,-25.2,42.0,67.2
109,RET-US-001-03558783,1,37.0,1,0.0,-22.2,37.0,59.2
110,RET-US-001-03558783,2,61.0,1,0.0,-36.6,61.0,97.6
114,RET-US-001-03558839,1,59.5,1,0.0,-59.5,59.5,119.0
139,RET-US-001-03558841,1,24.0,1,0.0,-24.0,24.0,48.0
155,RET-US-001-03558769,1,34.0,1,0.0,-20.4,34.0,54.4


In [38]:
transactions[
    [
        "Unit Price",
        "Quantity",
        "Discount",
        "Line Total",
        "Expected Line Total",
        "Line Total Difference"
    ]
].describe()

,Unit Price,Quantity,Discount,Line Total,Expected Line Total,Line Total Difference
count,6.416827e+06,6.416827e+06,6.416827e+06,6.416827e+06,6.416827e+06,6.416827e+06
mean,1.324640e+02,1.100243e+00,1.190578e-01,1.141912e+02,1.286456e+02,1.445479e+01
std,1.850971e+02,3.963792e-01,1.990280e-01,2.115865e+02,2.048277e+02,1.149429e+02
min,2.000000e+00,1.000000e+00,0.000000e+00,-3.348000e+03,1.400000e+00,0.000000e+00
25%,3.250000e+01,1.000000e+00,0.000000e+00,2.475000e+01,2.750000e+01,0.000000e+00
50%,5.100000e+01,1.000000e+00,0.000000e+00,4.350000e+01,4.690000e+01,0.000000e+00
75%,1.165000e+02,1.000000e+00,2.500000e-01,1.090000e+02,1.215000e+02,0.000000e+00
max,1.153500e+03,3.000000e+00,6.000000e-01,3.460500e+03,3.460500e+03,6.696000e+03


In [39]:
transactions["Discount"].value_counts().head(20)

Discount
0.00    4627961
0.50     899045
0.45     230070
0.35     220981
0.25     147943
0.20     133528
0.40     122473
0.60      34826
Name: count, dtype: int64

In [40]:
transactions["Transaction Type"].value_counts(dropna=False)

Transaction Type
Sale      6077200
Return     339627
Name: count, dtype: int64

In [41]:
transactions.groupby("Transaction Type").agg(
    Transactions=("Invoice ID", "count"),
    Total_Quantity=("Quantity", "sum"),
    Total_Line_Total=("Line Total", "sum"),
    Avg_Line_Total=("Line Total", "mean")
)

,Transactions,Total_Quantity,Total_Line_Total,Avg_Line_Total
Transaction Type,,,,
Return,339627,373947,-4.331565e+07,-127.538881
Sale,6077200,6686124,7.760605e+08,127.700343


In [42]:
financial_inconsistencies.groupby("Transaction Type").agg(
    Count=("Invoice ID", "count"),
    Total_Line_Total=("Line Total", "sum")
)

,Count,Total_Line_Total
Transaction Type,,
Return,339627,-43315647.63


In [43]:
pd.crosstab(
    transactions["Transaction Type"],
    transactions["Line Total"] < 0
)

Line Total,False,True
Transaction Type,,
Return,0,339627
Sale,6077200,0


In [44]:
returns = transactions[
    transactions["Transaction Type"].astype(str).str.upper().str.contains("RET")
].copy()

returns[
    [
        "Invoice ID",
        "Line",
        "Unit Price",
        "Quantity",
        "Discount",
        "Line Total",
        "Transaction Type"
    ]
].head(20)

,Invoice ID,Line,Unit Price,Quantity,Discount,Line Total,Transaction Type
15,RET-US-001-03558764,1,75.0,1,0.0,-45.0,Return
16,RET-US-001-03558764,2,45.5,1,0.0,-27.3,Return
31,RET-US-001-03558763,1,95.5,1,0.0,-95.5,Return
40,RET-US-001-03558767,1,54.0,1,0.0,-32.4,Return
103,RET-US-001-03558799,1,42.0,1,0.0,-25.2,Return
109,RET-US-001-03558783,1,37.0,1,0.0,-22.2,Return
110,RET-US-001-03558783,2,61.0,1,0.0,-36.6,Return
114,RET-US-001-03558839,1,59.5,1,0.0,-59.5,Return
139,RET-US-001-03558841,1,24.0,1,0.0,-24.0,Return
155,RET-US-001-03558769,1,34.0,1,0.0,-20.4,Return


In [45]:
returns[
    [
        "Unit Price",
        "Quantity",
        "Discount",
        "Line Total"
    ]
].describe()

,Unit Price,Quantity,Discount,Line Total
count,339627.000000,339627.000000,339627.0,339627.000000
mean,132.147786,1.101052,0.0,-127.538881
std,184.680598,0.398044,0.0,203.868898
min,2.000000,1.000000,0.0,-3348.000000
25%,32.500000,1.000000,0.0,-120.000000
50%,51.000000,1.000000,0.0,-46.500000
75%,115.500000,1.000000,0.0,-27.500000
max,1153.500000,3.000000,0.0,-1.400000


In [46]:
returns_analysis = returns.copy()

returns_analysis["Absolute Line Total"] = returns_analysis["Line Total"].abs()

returns_analysis["Return Ratio"] = (
    returns_analysis["Absolute Line Total"]
    / (returns_analysis["Unit Price"] * returns_analysis["Quantity"])
)

returns_analysis[
    [
        "Unit Price",
        "Quantity",
        "Discount",
        "Line Total",
        "Absolute Line Total",
        "Return Ratio"
    ]
].head(30)

,Unit Price,Quantity,Discount,Line Total,Absolute Line Total,Return Ratio
15,75.0,1,0.0,-45.0,45.0,0.6
16,45.5,1,0.0,-27.3,27.3,0.6
31,95.5,1,0.0,-95.5,95.5,1.0
40,54.0,1,0.0,-32.4,32.4,0.6
103,42.0,1,0.0,-25.2,25.2,0.6
109,37.0,1,0.0,-22.2,22.2,0.6
110,61.0,1,0.0,-36.6,36.6,0.6
114,59.5,1,0.0,-59.5,59.5,1.0
139,24.0,1,0.0,-24.0,24.0,1.0
155,34.0,1,0.0,-20.4,20.4,0.6


In [47]:
returns_analysis["Return Ratio"].value_counts().head(20)

Return Ratio
1.000000    240785
0.500000     49114
0.600000      5631
0.650000      5149
0.800000      4614
0.550000      4123
0.750000      4085
0.800000      3221
0.550000      2624
0.400000      1377
0.600000      1139
0.650000       997
0.400000       868
1.000000       284
1.000000       180
0.550141       127
0.650105       120
0.550204       120
0.550222       117
0.550120       117
Name: count, dtype: int64

In [50]:
sales = transactions[
    transactions["Transaction Type"].astype(str).str.upper() == "SALE"
].copy()

In [51]:
sales["Expected Line Total"] = (
    sales["Unit Price"]
    * sales["Quantity"]
    * (1 - sales["Discount"])
)

sales["Difference"] = (
    sales["Line Total"] - sales["Expected Line Total"]
).abs()

print(
    "Sales with inconsistency:",
    (sales["Difference"] > 0.01).sum()
)

Sales with inconsistency: 0


In [52]:
sales[
    sales["Difference"] > 0.01
][
    [
        "Invoice ID",
        "Line",
        "Unit Price",
        "Quantity",
        "Discount",
        "Line Total",
        "Expected Line Total",
        "Difference"
    ]
].head(20)

,Invoice ID,Line,Unit Price,Quantity,Discount,Line Total,Expected Line Total,Difference


In [53]:
print("Nombre d'incohérences dans les ventes :", len(
    sales[sales["Difference"] > 0.01]
))

Nombre d'incohérences dans les ventes : 0


## les 798 doublons

In [54]:
duplicates = transactions[
    transactions.duplicated(keep=False)
].copy()

print("Nombre de lignes concernées :", len(duplicates))
print("Nombre de doublons exacts :", transactions.duplicated().sum())

Nombre de lignes concernées : 1594
Nombre de doublons exacts : 798


In [55]:
duplicates.sort_values(
    ["Invoice ID", "Line"]
).head(20)

,Invoice ID,Line,Customer ID,Product ID,Size,Color,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,SKU,Transaction Type,Payment Method,Invoice Total,Expected Line Total,Line Total Difference
1697858,RET-CN-006-02871859,1,515701,1033,XXL,NaN,117.0,1,2023-02-10,0.0,-117.00,6,71,CNY,¥,MAUN1033-XXL-,Return,Credit Card,-117.00,117.0,234.00
1697966,RET-CN-006-02871859,1,515701,1033,XXL,NaN,117.0,1,2023-02-10,0.0,-117.00,6,71,CNY,¥,MAUN1033-XXL-,Return,Credit Card,-117.00,117.0,234.00
1708166,RET-CN-006-02878641,1,361248,4656,M,NEUTRAL,409.5,1,2023-03-14,0.0,-409.50,6,68,CNY,¥,FESW4656-M-NEUTRAL,Return,Credit Card,-409.50,409.5,819.00
1708291,RET-CN-006-02878641,1,361248,4656,M,NEUTRAL,409.5,1,2023-03-14,0.0,-409.50,6,68,CNY,¥,FESW4656-M-NEUTRAL,Return,Credit Card,-409.50,409.5,819.00
1718072,RET-CN-006-02885345,1,408627,5118,M,RED,401.5,1,2023-03-30,0.0,-260.98,6,69,CNY,¥,CHGI5118-M-RED,Return,Cash,-260.98,401.5,662.48
1718143,RET-CN-006-02885345,1,408627,5118,M,RED,401.5,1,2023-03-30,0.0,-260.98,6,69,CNY,¥,CHGI5118-M-RED,Return,Cash,-260.98,401.5,662.48
1718495,RET-CN-006-02885464,1,388779,4724,M,NaN,501.5,1,2023-03-31,0.0,-501.50,6,70,CNY,¥,MASP4724-M-,Return,Credit Card,-501.50,501.5,1003.00
1718668,RET-CN-006-02885464,1,388779,4724,M,NaN,501.5,1,2023-03-31,0.0,-501.50,6,70,CNY,¥,MASP4724-M-,Return,Credit Card,-501.50,501.5,1003.00
1718443,RET-CN-006-02885628,1,417288,5125,S,BEIGE,409.5,1,2023-03-31,0.0,-266.18,6,67,CNY,¥,FEDR5125-S-BEIGE,Return,Credit Card,-266.18,409.5,675.68
1718912,RET-CN-006-02885628,1,417288,5125,S,BEIGE,409.5,1,2023-03-31,0.0,-266.18,6,67,CNY,¥,FEDR5125-S-BEIGE,Return,Credit Card,-266.18,409.5,675.68


In [56]:
duplicates["Transaction Type"].value_counts()

Transaction Type
Return    1594
Name: count, dtype: int64

In [57]:
duplicate_counts = (
    transactions
    .groupby(list(transactions.columns), dropna=False)
    .size()
    .reset_index(name="Count")
)

duplicate_counts["Count"].value_counts().sort_index()

Count
1    6415233
2        794
3          2
Name: count, dtype: int64

In [58]:
duplicate_groups = duplicate_counts[
    duplicate_counts["Count"] > 1
]

print("Nombre de groupes de doublons :", len(duplicate_groups))
print("Nombre de lignes dans ces groupes :", 
      duplicate_groups["Count"].sum())

Nombre de groupes de doublons : 796
Nombre de lignes dans ces groupes : 1594


In [59]:
transactions_clean = transactions.drop_duplicates().copy()

In [60]:
print("Avant :", len(transactions))
print("Après :", len(transactions_clean))
print("Doublons restants :", transactions_clean.duplicated().sum())

Avant : 6416827
Après : 6416029
Doublons restants : 0


In [61]:
transactions_raw = transactions.copy()

transactions_clean = transactions.drop_duplicates().copy()

## Analyse des valeurs manquantes

In [62]:
datasets = {
    "customers": customers,
    "products": products,
    "discounts": discounts,
    "employees": employees,
    "stores": stores,
    "transactions": transactions_clean
}

for name, df in datasets.items():
    print(f"\n===== {name.upper()} =====")
    
    missing = pd.DataFrame({
        "Missing": df.isna().sum(),
        "Missing %": (df.isna().mean() * 100).round(2)
    })
    
    print(missing[missing["Missing"] > 0].sort_values(
        "Missing", ascending=False
    ))


===== CUSTOMERS =====
           Missing  Missing %
Job Title   584185      35.55

===== PRODUCTS =====
       Missing  Missing %
Color    12445      69.37
Sizes     2070      11.54

===== DISCOUNTS =====
              Missing  Missing %
Category           10       5.52
Sub Category       10       5.52

===== EMPLOYEES =====
Empty DataFrame
Columns: [Missing, Missing %]
Index: []

===== STORES =====
Empty DataFrame
Columns: [Missing, Missing %]
Index: []

===== TRANSACTIONS =====
       Missing  Missing %
Color  4350231      67.80
Size    413049       6.44


In [63]:
product_color_map = (
    products[["Product ID", "Color"]]
    .drop_duplicates("Product ID")
)

transactions_color_check = transactions_clean.merge(
    product_color_map,
    on="Product ID",
    how="left",
    suffixes=("", "_Product")
)

print("Transactions avec Color manquant :",
      transactions_color_check["Color"].isna().sum())

print("Parmi elles, couleur disponible dans Products :",
      (
          transactions_color_check["Color"].isna()
          & transactions_color_check["Color_Product"].notna()
      ).sum())

Transactions avec Color manquant : 4350231
Parmi elles, couleur disponible dans Products : 0


In [64]:
color_missing_by_type = (
    transactions_clean
    .groupby("Transaction Type")["Color"]
    .apply(lambda x: x.isna().sum())
)

print(color_missing_by_type)

Transaction Type
Return     230045
Sale      4120186
Name: Color, dtype: int64


In [65]:
color_rate_by_type = (
    transactions_clean
    .groupby("Transaction Type")["Color"]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
)

print(color_rate_by_type)

Transaction Type
Return    67.89
Sale      67.80
Name: Color, dtype: float64


### Analyse des valeurs manquantes - Color

La colonne `Color` présente 4 350 231 valeurs manquantes dans les transactions,
soit 67,80 % des transactions.

La proportion est similaire selon le type de transaction :
- Sale : 67,80 %
- Return : 67,89 %

Une tentative de récupération de la couleur depuis la table `Products`
via `Product ID` n'a permis de récupérer aucune valeur.

Les valeurs manquantes ne seront donc pas supprimées ni imputées à partir
d'une autre table. Leur traitement sera effectué lors de l'étape de nettoyage.

In [66]:
size_missing_by_type = (
    transactions_clean
    .groupby("Transaction Type")["Size"]
    .apply(lambda x: x.isna().sum())
)

print(size_missing_by_type)

size_rate_by_type = (
    transactions_clean
    .groupby("Transaction Type")["Size"]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
)

print(size_rate_by_type)

Transaction Type
Return     21947
Sale      391102
Name: Size, dtype: int64
Transaction Type
Return    6.48
Sale      6.44
Name: Size, dtype: float64


In [68]:
product_sizes = (
    products[["Product ID", "Sizes"]]
    .drop_duplicates("Product ID")
)

transactions_size_check = transactions_clean.merge(
    product_sizes,
    on="Product ID",
    how="left"
)

missing_size = transactions_size_check["Size"].isna()

print("Size manquant dans Transactions :", missing_size.sum())

print(
    "Parmi eux, Products possède une information Sizes :",
    (
        missing_size
        & transactions_size_check["Sizes"].notna()
    ).sum()
)

Size manquant dans Transactions : 413049
Parmi eux, Products possède une information Sizes : 0


In [69]:
print("Nombre total de clients :", len(customers))
print("Job Title manquant :", customers["Job Title"].isna().sum())

print("\nRépartition des Job Title renseignés :")
print(customers["Job Title"].value_counts().head(20))

Nombre total de clients : 1643306
Job Title manquant : 584185

Répartition des Job Title renseignés :
Job Title
Museum/gallery exhibitions officer      1764
Theme park manager                      1756
Designer, industrial/product            1754
Engineer, agricultural                  1753
Designer, jewellery                     1748
Teacher, secondary school               1747
Information systems manager             1747
Technical brewer                        1746
Outdoor activities/education manager    1744
Social worker                           1744
Professor Emeritus                      1742
Surveyor, building control              1740
Scientist, marine                       1740
Ranger/warden                           1739
Heritage manager                        1739
Chartered loss adjuster                 1738
Ship broker                             1736
Academic librarian                      1735
Actor                                   1735
Tax inspector                    

In [70]:
print("\nJob Title manquant selon le pays :")
print(
    customers.groupby("Country")["Job Title"]
    .apply(lambda x: x.isna().sum())
    .sort_values(ascending=False)
    .head(20)
)


Job Title manquant selon le pays :
Country
中国                207498
United States     159904
United Kingdom     68960
Deutschland        52832
France             39165
España             38844
Portugal           16982
Name: Job Title, dtype: int64


### Analyse des valeurs manquantes - Customers

La colonne `Job Title` contient 584 185 valeurs manquantes sur
1 643 306 clients, soit 35,55 %.

Les valeurs renseignées présentent une grande diversité de métiers.
Aucune catégorie dominante ne permet une imputation fiable des valeurs
manquantes.

Les valeurs manquantes ne justifient pas la suppression des clients.
Lors de l'étape de nettoyage, elles seront remplacées par `Unknown`
afin de conserver l'ensemble des clients sans introduire une valeur
de métier arbitraire.

In [71]:
print("Produits :", len(products))
print("Color manquant :", products["Color"].isna().sum())
print("Sizes manquant :", products["Sizes"].isna().sum())

print("\nCatégories des produits concernés par Color manquant :")
print(
    products[products["Color"].isna()]["Category"]
    .value_counts()
)

print("\nSous-catégories des produits concernés par Color manquant :")
print(
    products[products["Color"].isna()]["Sub Category"]
    .value_counts().head(20)
)

Produits : 17940
Color manquant : 12445
Sizes manquant : 2070

Catégories des produits concernés par Color manquant :
Category
Feminine     5722
Masculine    4376
Children     2347
Name: count, dtype: int64

Sous-catégories des produits concernés par Color manquant :
Sub Category
Accessories                             1933
Sportswear                              1380
Pants and Jeans                         1350
Shirts and Blouses                       690
T-shirts and Tops                        690
Suits and Sets                           690
Lingerie and Pajamas                     690
Underwear and Pajamas                    690
Shirts                                   686
Skirts and Shorts                        683
Pajamas                                  682
Suits and Blazers                        679
Sweaters                                 678
Coats and Blazers                        219
Girl and Boy (1-5 years, 6-14 years)     115
T-shirts and Polos                       110

In [72]:
print("Produits avec Sizes manquant :", products["Sizes"].isna().sum())

print("\nCatégories :")
print(
    products[products["Sizes"].isna()]["Category"]
    .value_counts()
)

print("\nSous-catégories :")
print(
    products[products["Sizes"].isna()]["Sub Category"]
    .value_counts().head(20)
)

Produits avec Sizes manquant : 2070

Catégories :
Category
Feminine     690
Masculine    690
Children     690
Name: count, dtype: int64

Sous-catégories :
Sub Category
Accessories    2070
Name: count, dtype: int64


In [73]:
missing_discounts = discounts[
    discounts["Category"].isna() |
    discounts["Sub Category"].isna()
]

print(missing_discounts)

         Start        End  Discont  \
32  2020-11-27 2020-11-27      0.6   
33  2020-12-20 2020-12-31      0.5   
66  2021-11-27 2021-11-27      0.6   
67  2021-12-20 2021-12-31      0.5   
100 2022-11-27 2022-11-27      0.6   
101 2022-12-20 2022-12-31      0.5   
134 2023-11-27 2023-11-27      0.6   
135 2023-12-20 2023-12-31      0.5   
168 2024-11-27 2024-11-27      0.6   
169 2024-12-20 2024-12-31      0.5   

                                        Description Category Sub Category  
32   60% discount during our Black Friday Mega Sale      NaN          NaN  
33      50% discount during our Holiday Season Sale      NaN          NaN  
66   60% discount during our Black Friday Mega Sale      NaN          NaN  
67      50% discount during our Holiday Season Sale      NaN          NaN  
100  60% discount during our Black Friday Mega Sale      NaN          NaN  
101     50% discount during our Holiday Season Sale      NaN          NaN  
134  60% discount during our Black Friday Mega Sa

In [74]:
print("Nombre de lignes :", len(missing_discounts))

print("\nDates :")
print(missing_discounts[["Start", "End"]])

print("\nDescriptions :")
print(missing_discounts["Description"].to_string(index=False))

Nombre de lignes : 10

Dates :
         Start        End
32  2020-11-27 2020-11-27
33  2020-12-20 2020-12-31
66  2021-11-27 2021-11-27
67  2021-12-20 2021-12-31
100 2022-11-27 2022-11-27
101 2022-12-20 2022-12-31
134 2023-11-27 2023-11-27
135 2023-12-20 2023-12-31
168 2024-11-27 2024-11-27
169 2024-12-20 2024-12-31

Descriptions :
60% discount during our Black Friday Mega Sale
   50% discount during our Holiday Season Sale
60% discount during our Black Friday Mega Sale
   50% discount during our Holiday Season Sale
60% discount during our Black Friday Mega Sale
   50% discount during our Holiday Season Sale
60% discount during our Black Friday Mega Sale
   50% discount during our Holiday Season Sale
60% discount during our Black Friday Mega Sale
   50% discount during our Holiday Season Sale
